# Phase 38 -- full evaluation over `eval_all_v1.jsonl` (108 questions/conversations)

First run of every eval set combined in one pass, under today's complete production
pipeline (Claude Haiku 4.5, compact prompt, query reparser rewritten this session,
rule 4/10/12 rewrites, heading-typography fix -- none of which have been measured
together before). Covers:

- **eval_v3** (60) -- standard faithfulness/relevancy judge (`JUDGE_PROMPT`, kept
  byte-identical to every historical run for comparability) + citation regex proxy.
- **eval_calc_v4** (14) -- citation proxy + whether the expected calculator fired
  (`res["calculation"]` populated). Not the full narration-correctness check Phase
  25/32 built (that needs bespoke per-question expected values); this is a coarser
  structural pass, stated as a scope limit rather than claimed as equivalent.
- **eval_rates** (10) -- retrieval only (its original purpose: did the funding
  family get retrieved), no generation grading.
- **eval_mechanisms_v1** (16) -- the new set built this session. m01-m10 (scope
  exclusions, out-of-scope, judgment questions) have no single correct citation to
  check against, so they're graded with the new `BEHAVIOR_JUDGE_PROMPT` against
  each question's `expected_behavior` field. m11-m16 use the citation proxy.
- **eval_multiturn_v1** (8 conversations) -- history-aware retrieval via
  `rewrite_for_retrieval`, then the behavior judge against `expected_behavior`
  (same field/mechanism eval_mechanisms_v1 uses). One trial per conversation in
  this pass, not Phase 32's three -- stated as a scope limit; a discrepancy here is
  grounds for repeat trials before concluding anything, not a final answer.

**Known caveat, carried forward from Phase 31/CLAUDE.md, not solved here:** the
standard judge (`claude-sonnet-5`) shares a vendor with the production model
(`claude-haiku-4.5`), a possible self-preference bias. Used anyway because it is
the ONLY judge with historical comparability to every prior score in this project
-- switching would make this run incomparable to its own baseline, trading one
open problem for a different one. Read as "measured with the same known bias this
project has always had," not as bias-free.


In [1]:
import sys
import json
from pathlib import Path

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
sys.path.insert(0, str(PROJECT))

from dotenv import load_dotenv
load_dotenv(PROJECT / ".env")

import pandas as pd
import collections

from src.retrieval import Retriever
from src.generation import (get_production_generator, PRODUCTION_MODEL_ID,
                             extract_cited_articles, grounded_article_numbers)
from evallib.metrics import metrics
from evallib.judge import (JUDGE_MODEL, JUDGE_MAX_TOKENS, JUDGE_PROMPT, make_judge,
                            BEHAVIOR_JUDGE_PROMPT, make_behavior_judge,
                            format_context_for_judge)

EVAL_DIR = PROJECT / "eval"
print("production model:", PRODUCTION_MODEL_ID)
print("judge model:", JUDGE_MODEL)

eval_all = []
with open(EVAL_DIR / "eval_all_v1.jsonl", encoding="utf-8") as f:
    for line in f:
        eval_all.append(json.loads(line))

print(f"{len(eval_all)} rows")
print(collections.Counter(r["source_set"] for r in eval_all))


production model: anthropic/claude-haiku-4.5
judge model: anthropic/claude-sonnet-5
108 rows
Counter({'eval_v3': 60, 'eval_mechanisms_v1': 16, 'eval_calc_v4': 14, 'eval_rates': 10, 'eval_multiturn_v1': 8})


In [2]:
RESULTS_PATH = EVAL_DIR / "phase38_full_eval_all_v1.csv"

retriever = Retriever()
gen = get_production_generator()
judge = make_judge(gen.client, max_tokens=JUDGE_MAX_TOKENS)
behavior_judge = make_behavior_judge(gen.client, max_tokens=JUDGE_MAX_TOKENS)

rows = []
if RESULTS_PATH.exists():
    rows = pd.read_csv(RESULTS_PATH, encoding="utf-8-sig").to_dict("records")
done = {r["question_id"] for r in rows}
print(f"resuming: {len(done)} already done")

for d in eval_all:
    qid = d["question_id"]
    if qid in done:
        continue

    q = d["question_text"]
    history = d.get("history")
    is_multiturn = isinstance(history, list) and len(history) > 0
    gold_law = d.get("answer_law")
    gold_arts = set(d.get("answer_article_numbers") or [])
    has_gold = gold_law is not None and len(gold_arts) > 0

    # --- retrieval ---
    if is_multiturn:
        retrieval_query = gen.rewrite_for_retrieval(q, history)
    else:
        retrieval_query = q
    chunks = retriever.retrieve(retrieval_query, top_k=5)

    if has_gold:
        retrieved_same_law = [c["article_number"] for c in chunks if c["law"] == gold_law]
        rank = next((i + 1 for i, a in enumerate(retrieved_same_law) if a in gold_arts), None)
        full_coverage = gold_arts.issubset(set(retrieved_same_law))
    else:
        rank, full_coverage = None, None

    # --- generation ---
    res = gen.generate(q, chunks, history=history if is_multiturn else None)
    answer = res["answer"]
    calc_fired = res.get("calculation") is not None

    # --- citation proxy (only meaningful when there's a gold article) ---
    if has_gold:
        cited = extract_cited_articles(answer)
        grounded = grounded_article_numbers(chunks)
        fabricated = cited - grounded
        correct = bool(gold_arts & cited)
        all_correct = gold_arts.issubset(cited)
    else:
        cited, fabricated, correct, all_correct = set(), set(), None, None

    # --- grading: behavior judge (if expected_behavior given) else standard judge
    #     only for eval_v3 (its historical calibration set) else no LLM judge ---
    expected_behavior = d.get("expected_behavior")
    judge_kind, judge_result = None, None
    try:
        if expected_behavior:
            judge_kind = "behavior"
            judge_result = behavior_judge(q, chunks, answer, expected_behavior)
        elif d["source_set"] == "eval_v3":
            judge_kind = "standard"
            judge_result = judge(q, chunks, answer)
    except Exception as e:
        judge_result = {"error": f"{type(e).__name__}: {str(e)[:200]}"}

    rows.append({
        "question_id": qid, "source_set": d["source_set"], "question": q,
        "gold_law": gold_law, "gold_articles": sorted(gold_arts),
        "rank": rank, "full_coverage": full_coverage,
        "cited_articles": sorted(cited), "fabricated": sorted(fabricated),
        "correct": correct, "all_correct": all_correct,
        "calc_fired": calc_fired,
        "judge_kind": judge_kind, "judge_result": json.dumps(judge_result, ensure_ascii=False) if judge_result else None,
        "answer_len": len(answer), "answer": answer,
    })
    pd.DataFrame(rows).to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")
    print(f"  {qid} ({d['source_set']}): rank={rank} correct={correct} judge={judge_kind}:{judge_result}")

results_df = pd.DataFrame(rows)
print(f"\n{len(results_df)}/{len(eval_all)} done")


resuming: 0 already done


  q01 (eval_v3): rank=2 correct=True judge=standard:{'faithfulness_score': 4, 'faithfulness_reason': "All claims are supported by the cited articles, though the statement that 'the Corporation pays from day 4 onward' is a slight inference not explicitly stated in Article 29.", 'relevancy_score': 4, 'relevancy_reason': 'The answer thoroughly covers medical care, disability allowances/pensions, and death benefits relevant to the injury question, but omits the funeral expenses benefit mentioned in Article 25(d).'}


  q02 (eval_v3): rank=1 correct=True judge=standard:{'faithfulness_score': 4, 'faithfulness_reason': 'The core claims about prosthetics and transport expenses are directly quoted from Article 26, but the assumption that the questioner has a work injury (a precondition for these benefits) is not established or supported by the documents.', 'relevancy_score': 4, 'relevancy_reason': 'The answer directly addresses both the artificial limb and travel cost questions with relevant citations, though it fails to note that this coverage applies specifically to work-related injuries, which is an important qualifier.'}


  q03 (eval_v3): rank=1 correct=True judge=standard:{'faithfulness_score': 3, 'faithfulness_reason': "Most claims are supported, but it inaccurately attributes the police-report requirement to Article 27(a) as unconditional (SSL says only injuries the Board decides must be reported) and merges labor96 87(a)'s death/incapacitating-injury condition without noting it applies only in those cases.", 'relevancy_score': 5, 'relevancy_reason': 'The answer directly lists the immediate actions required after a workplace injury, matching the question comprehensively.'}


  q04 (eval_v3): rank=1 correct=True judge=standard:{'faithfulness_score': 5, 'faithfulness_reason': 'All claims (75%/65% rates, treatment conditions, first-3-days employer payment, calculation basis, Article 86 scope) are directly supported by the cited provisions.', 'relevancy_score': 5, 'relevancy_reason': 'The answer directly addresses whether and how much the user will be paid during work-injury recovery, covering both applicable legal regimes with clear conditions.'}


...

  mt04 (eval_multiturn_v1): rank=6 correct=False judge=behavior:{'pass': True, 'reason': 'The assistant again presented both male and female cases separately rather than assuming a sex from its own earlier recitation, correctly treating sex as still unknown.'}


  mt05 (eval_multiturn_v1): rank=1 correct=True judge=behavior:{'pass': False, 'reason': "The assistant correctly cited Articles 57/58/59 but then improperly added a clarifying personal question ('هل تشغل منصباً إشرافياً أو إدارياً...') instead of answering purely institutionally."}


  mt06 (eval_multiturn_v1): rank=1 correct=True judge=behavior:{'pass': True, 'reason': 'The assistant explicitly presented both outcomes based on sex (male vs female thresholds) for the 230-subscription case rather than assuming or skipping the sex question, satisfying the required behavior.'}


  mt07 (eval_multiturn_v1): rank=1 correct=True judge=behavior:{'pass': True, 'reason': 'The assistant correctly retrieved all funding-family articles (59, 24, 42, 48) and used the salary of 1000 JD from the prior turn to compute a real employer contribution of 142.50 JD, showing the salary was successfully folded into the query and extracted.'}


  mt08 (eval_multiturn_v1): rank=1 correct=True judge=behavior:{'pass': True, 'reason': "The assistant directly answered using Article 27's explicit Hajj leave protection without asking any clarifying question."}

108/108 done


## Aggregate results, by source set

In [3]:
for src in results_df["source_set"].unique():
    sub = results_df[results_df["source_set"] == src]
    print(f"=== {src} (n={len(sub)}) ===")
    ranked = sub.dropna(subset=["rank"])
    if len(ranked):
        print("  retrieval:", metrics(ranked, col="rank", ndigits=3, full_coverage=True))
    n_scored = sub["correct"].notna().sum()
    if n_scored:
        print(f"  correct-article-cited: {sub['correct'].sum()}/{n_scored}")
        print(f"  fully complete: {sub['all_correct'].sum()}/{n_scored}")
        print(f"  fabrication flags: {(sub['fabricated'].apply(lambda x: len(eval(x)) if isinstance(x, str) else 0) > 0).sum()}/{n_scored}")
    if src == "eval_calc_v4":
        print(f"  calculator fired: {sub['calc_fired'].sum()}/{len(sub)}")
    print(f"  mean answer length: {sub['answer_len'].mean():.0f} chars")
    print()


=== eval_v3 (n=60) ===
  retrieval: {'n': 60, 'recall@1': np.float64(0.917), 'recall@5': np.float64(1.0), 'recall@10': np.float64(1.0), 'mrr': np.float64(0.953), 'full_coverage@10': 0.967}
  correct-article-cited: 59/60
  fully complete: 57/60
  fabrication flags: 0/60
  mean answer length: 1039 chars

=== eval_calc_v4 (n=14) ===
  retrieval: {'n': 14, 'recall@1': np.float64(0.929), 'recall@5': np.float64(1.0), 'recall@10': np.float64(1.0), 'mrr': np.float64(0.964), 'full_coverage@10': 1.0}
  correct-article-cited: 14/14
  fully complete: 11/14
  fabrication flags: 0/14
  calculator fired: 14/14
  mean answer length: 644 chars

=== eval_rates (n=10) ===
  retrieval: {'n': 10, 'recall@1': np.float64(0.9), 'recall@5': np.float64(1.0), 'recall@10': np.float64(1.0), 'mrr': np.float64(0.92), 'full_coverage@10': 1.0}
  correct-article-cited: 8/10
  fully complete: 8/10
  fabrication flags: 0/10
  mean answer length: 708 chars

=== eval_mechanisms_v1 (n=16) ===
  retrieval: {'n': 13, 'recall@

## Standard judge (eval_v3 only) -- faithfulness / relevancy

In [4]:
v3 = results_df[results_df["source_set"] == "eval_v3"].copy()
def _parse(s):
    try:
        return json.loads(s) if isinstance(s, str) else {}
    except Exception:
        return {}
v3_judged = v3["judge_result"].apply(_parse)
v3["faithfulness"] = v3_judged.apply(lambda d: d.get("faithfulness_score"))
v3["relevancy"] = v3_judged.apply(lambda d: d.get("relevancy_score"))

print("faithfulness mean:", v3["faithfulness"].mean(), "| min:", v3["faithfulness"].min())
print("relevancy mean:", v3["relevancy"].mean(), "| min:", v3["relevancy"].min())
print()
print("faithfulness distribution:\n", v3["faithfulness"].value_counts().sort_index())
print()
print("relevancy distribution:\n", v3["relevancy"].value_counts().sort_index())


faithfulness mean: 4.65 | min: 3
relevancy mean: 4.833333333333333 | min: 2

faithfulness distribution:
 faithfulness
3     5
4    11
5    44
Name: count, dtype: int64

relevancy distribution:
 relevancy
2     1
4     7
5    52
Name: count, dtype: int64


## Behavior judge (eval_mechanisms_v1 m01-m10 + eval_multiturn_v1) -- pass/fail

In [5]:
behavior_rows = results_df[results_df["judge_kind"] == "behavior"].copy()
behavior_parsed = behavior_rows["judge_result"].apply(_parse)
behavior_rows["pass"] = behavior_parsed.apply(lambda d: d.get("pass"))
behavior_rows["reason"] = behavior_parsed.apply(lambda d: d.get("reason"))

print(f"{behavior_rows['pass'].sum()}/{len(behavior_rows)} passed")
print()
for _, r in behavior_rows.iterrows():
    print(f"{r['question_id']} ({r['source_set']}): pass={r['pass']} -- {r['reason']}")


16/18 passed

m01 (eval_mechanisms_v1): pass=True -- The answer correctly states that Labour Law 1996 Article 3(a) excludes public servants/municipal workers, so it does not apply to the government ministry employee.
m02 (eval_mechanisms_v1): pass=True -- The answer correctly states that Labor Law 1996 Article 3(a) excludes government employees, and does not treat normal labor law protections as applicable.
m03 (eval_mechanisms_v1): pass=True -- The answer correctly states domestic workers are excluded under Article 3(b) and governed by a separate regulation instead of standard labor law provisions.
m04 (eval_mechanisms_v1): pass=True -- الإجابة ذكرت صراحة أن العاملين في المنازل مستثنون بموجب المادة 3(ب) من قانون العمل ويخضعون لنظام منفصل بدلاً من الأحكام الاعتيادية.
m05 (eval_mechanisms_v1): pass=True -- The answer states that the law only covers professions added by Council of Ministers decision per Article 3 and that an unlisted profession is not covered.
m06 (eval_mechanisms_v1): p

## Manual read of flagged/failed cases

Read every wrong/incomplete/fabrication-flagged/behavior-failed case by hand before
concluding anything -- this project's repeated finding (Phase 20/21/34) is that a
raised flag often turns out to be a scorer artifact (in-text cross-references,
wording variants) rather than a real defect.

In [6]:
wrong = results_df[results_df["correct"] == False]
incomplete = results_df[(results_df["correct"] == True) & (results_df["all_correct"] == False)]
flagged = results_df[results_df["fabricated"].apply(lambda x: len(eval(x)) if isinstance(x, str) else 0) > 0]
behavior_failed = results_df[(results_df["judge_kind"] == "behavior") &
                              (results_df["judge_result"].apply(lambda s: _parse(s).get("pass")) == False)]

print(f"{len(wrong)} wrong (gold article never cited):", list(wrong["question_id"]))
print(f"{len(incomplete)} partially correct:", list(incomplete["question_id"]))
print(f"{len(flagged)} flagged fabrication candidates:", list(flagged["question_id"]))
print(f"{len(behavior_failed)} behavior judge failures:", list(behavior_failed["question_id"]))


6 wrong (gold article never cited): ['q10', 'r04', 'r10', 'm13', 'm14', 'mt04']
6 partially correct: ['q31', 'q56', 'c05', 'c06', 'c07', 'm12']
0 flagged fabrication candidates: []
2 behavior judge failures: ['m10', 'mt05']


In [7]:
to_read = pd.concat([wrong, incomplete, flagged, behavior_failed]).drop_duplicates(subset="question_id")
for _, r in to_read.iterrows():
    print("=" * 90)
    print(r["question_id"], r["source_set"], "-- gold:", r["gold_articles"], "| cited:", r["cited_articles"],
          "| fabricated:", r["fabricated"], "| judge:", r["judge_result"])
    print(r["question"])
    print()
    print(r["answer"][:1200])
    print()


q10 eval_v3 -- gold: ['44', '45'] | cited: ['4', '43'] | fabricated: [] | judge: {"faithfulness_score": 3, "faithfulness_reason": "The conditions cited (Article 43(b), Article 44, Article 4) are accurately paraphrased, but the answer omits the actual payment formula from Article 45 (allowance equal to last wage subject to deduction) despite having that information available.", "relevancy_score": 2, "relevancy_reason": "Instead of explaining eligibility criteria and stating the payment amount (full last wage per Article 45), the assistant only asks clarifying questions and defers the actual answer, leaving the question substantially unanswered."}
I am pregnant and working. Do I qualify for a maternity payment, and how much would it be?

To answer this, I need a few facts:

1. **How long have you been insured?** Do you have at least 6 continuous or discontinuous contributions to Maternity Insurance in the 12 months before your maternity leave begins, or have you been subject to the Mater

## Verdict

**Note before the numbers: `metrics()` was called on a pre-`dropna()`'d rank column
in the aggregate cell above, which silently drops real misses from the recall@k
denominator and inflates it.** Recomputed correctly below (passing the full
has-gold subset, NaN ranks included) -- this is the number that actually matters:

| set | recall@1 | recall@5 | mrr | full_coverage |
|---|---|---|---|---|
| eval_v3 (n=60) | .917 | 1.000 | .953 | .967 |
| eval_calc_v4 (n=14) | .929 | 1.000 | .964 | 1.000 |
| eval_rates (n=10) | .900 | 1.000 | .920 | 1.000 |
| eval_mechanisms_v1 (n=14 w/ gold) | .643 | **.929** | .750 | .929 |
| eval_multiturn_v1 (n=8) | .750 | **.750** | .792 | 1.000 |

Also: `full_coverage@10` / the printed "recall@10" is mislabeled throughout --
retrieval was only run `top_k=5`, so per `evallib/metrics.py`'s own documented
caveat, that column is actually recall@5 again, not a deeper search.

**eval_v3:** recall@1 .917 vs the Phase 31/34 baseline of .883 -- an *improvement*,
not a regression, and plausibly just Pinecone's approximate-NN variance (documented
before, e.g. Phase 25's worst-rank drift). Judge: faithfulness 4.65 (min 3 x5),
relevancy 4.83 (min 2 x1) -- both in the same range as every prior full run, no
regression. Zero fabrications, 59/60 correct-article-cited.

**eval_rates:** the "8/10 correct-article-cited" is NOT a retrieval regression --
every one of the 10 questions had its funding-family article at rank 1 except r08
(rank 5, still found) -- 10/10 by the metric this set was actually built to
validate. The 2 citation misses are read as: **r04** is a question-ambiguity
artifact (the model answered "does my employer pay me compensation" instead of the
intended "does my employer fund the insurance", both defensible readings of the
Arabic question); **r10** is a real, minor completeness gap -- the model correctly
asks for missing nationality/age facts before quoting a rate, but never states what
the rate *would be* once those conditions are met, unlike the pattern calc.py's own
calculators use elsewhere (state the figure, ask/condition on what's missing,
don't withhold both).

**eval_calc_v4:** calculator fired 14/14, zero fabrications. The 3 "incomplete"
citations (c05-c07) all omit Article 62 while correctly answering an
early-retirement-only question with Article 64 -- per this project's own standing
rule ("completeness means relevant, not exhaustive"), this is plausibly correct,
not a defect: the question never asks about normal retirement age. Not certain
enough to say the original v4 ground truth is wrong outright, but the answers read
as reasonable.

**eval_mechanisms_v1 (this session's new set):** rules 2 (out-of-scope), 13
(judgment), and 22 (scope exclusions) are confirmed working in practice, not just
in isolated spot checks -- 8/9 direct behavior-judge passes on m01-m09, plus the
citation-based m01-m06 all correct. **m13/m14 "wrong citation" is a flaw in my own
eval design, not a model defect:** I set the gold article to the precondition
article (labor96 Art 86) itself, but a correct answer naturally cites the
*substantive* payer articles (SSL 29/30, labor96 88/90/92) instead of the scoping
clause -- the model's actual behavior (correctly splitting "who pays" by SSL
coverage) is right in both cases; the ground truth should have targeted the
substantive articles, or this pair should move to behavior-judge grading like
m07-m10. **m14's retrieval genuinely missed rank <=5** (Article 86 not in top-5 for
the Arabic phrasing) -- real, if narrow: recall@5 for mechanisms is .929, not
1.000, driven by this one case. **m10 (judgment) failed on a real, minor gap:** it
gave a good neutral benefit-vs-cost framing but never stated the 17.5% headline
number my `expected_behavior` required. **m12 (overtime, self-identified shift
supervisor) is a genuine precision question, not settled either way:** the model
confidently ruled the person excluded from overtime under Article 58, but Article
58's "general supervision of the institution" is arguably a higher bar than "shift
supervisor" -- a plausible over-exclusion, flagged rather than fixed.

**eval_multiturn_v1: one real, newly-discovered feature conflict.** mt05 is Phase
32's "no over-asking" control (a general institutional question must not trigger a
clarifying personal question) -- and it now fails, because Phase 28's overtime
calculator guardrail asks about supervisory/travel status **unconditionally**
whenever Article 58 is retrieved, with no check for whether the question is
personal ("am I entitled") or general ("what are the exceptions"). These two
features were built in different phases and never tested together until this run.
**This is the one finding from this whole pass worth actually fixing**, not just
noting -- everything else here is either an improvement, noise, or a scorer/eval-
design artifact. mt03/mt04's rank=6 (Article 64 outside the top-5 window) did not
cost correctness -- both turns still produced the right figures, evidently either
via family-expansion mechanics or from the model correctly reusing figures already
established earlier in the same conversation; mt04's "0 citations" is a citation-
regex artifact on a rapid arithmetic follow-up, not a grounding failure (verified
by reading the answer: 252-200=52, 228-200=28, both exactly right).

**Bottom line:** nothing regressed against any historical baseline. Zero
fabrications across all 108. The new eval_mechanisms_v1 set did its job -- it
confirmed rules 2/13/22 work, and it surfaced two real, previously-unknown issues
(the mt05/Article-58 guardrail conflict, and the m10 missing-number gap) that no
prior eval pass could have caught, since neither the overtime guardrail nor the
judgment-question rule had ever been tested together with multi-turn or the
behavior judge before today.

## Post-fix re-check (not a full re-run)

Two fixes applied after the verdict above: (1) `src/calc.py`'s overtime guardrail
now only asks the supervisor/travel question when the question is about the
person's own eligibility, not a general "what are the rules" question; (2) m13/m14
switched from citation-check to behavior-judge grading (the citation check was
testing the wrong thing -- see verdict above). Only the affected rows were re-run,
not the full 108 -- nothing else changed.

- **mt05** (the real bug): regenerated fresh and re-judged. **Now passes** --
  "The assistant directly answered from Articles 57/58 (and additionally 75)
  without asking any clarifying personal question." Confirmed on 3 additional
  manual trials outside this notebook too (0/3 asked).
- **m12** (personal question, self-identified supervisor): re-verified on 2 fresh
  trials -- still correctly answers directly without re-asking, confirming the fix
  didn't break the personal-question path.
- **m13**: re-judged against the same stored answer with the new
  `expected_behavior`. **Now passes** -- correctly names SS coverage as the
  deciding condition and both payers.
- **m14** (the Arabic version of the same question): re-judged, **still fails**,
  but now for a real reason instead of a bad gold target -- the judge found the
  Arabic answer presents the Corporation and the employer as simultaneous payers
  for different injury stages, without ever stating the SS-coverage condition that
  decides which one applies (unlike m13's English answer, which does). A genuine,
  language-specific gap, not fixed here -- worth a closer look if this pattern
  matters (Article 86 possibly needs the same explicit "who pays" framing check in
  Arabic that Phase 28's rule 21 was built for in English).

Updated behavior-judge tally: **18/20** (up from 16/18 pre-fix; m10's fail is
unchanged and undiagnosed further, m14 now fails for the right reason instead of
the citation check failing for the wrong one).

## m14 fixed too

Root cause: `_ensure_preconditions`'s labor96/Article-86 rule required 2 of
{Article 90, Article 91} within `PRECONDITION_TRIGGER_DEPTH` (37) before adding
Article 86. For m14's Arabic phrasing, only Article 90 made it into that window --
Article 91 (a narrow provision about task-work-basis compensation calculation, not
really central to "who pays") ranked outside it for Arabic even though it ranked
close enough for the English phrasing of the same question (m13). `trigger_count`
lowered from 2 to 1 for this rule -- Article 90 alone is the article that actually
matters here.

**Verified:**
- Article 86 now appears in m14's retrieved context (rank 4, matching m13's rank).
- **Full `eval_v3` retrieval re-run: bit-identical to the pre-fix baseline**
  (recall@1 .917, recall@5 1.000, MRR .953, full_coverage .967) -- zero regression
  from loosening a mechanism shared across every question in the corpus, not just
  this one case.
- m14 regenerated fresh and re-judged 3 times: **3/3 pass**, matching m13's
  behavior -- correctly names Social Security coverage as the deciding condition
  and both payers.

Updated behavior-judge tally: **19/20** (m10 is the only remaining fail, unchanged
and undiagnosed further this session).

## m10 fixed too -- all behavior-judge findings from this run now closed

Root cause was retrieval, not the prompt: ssl Article 7 (who may join voluntary
Old-Age/Disability/Death insurance, and that they pay both shares in full) never
states the actual rate -- that number (17.5%) is in Article 59(a)(3), a different
article about contribution funding generally, and a plain "should I join
voluntarily" question never retrieved it. The model correctly described the
obligation qualitatively from Article 7 alone; it simply never had the number to
quantify it with. New `PRECONDITION_ARTICLES` rule added: ssl Article 59 as anchor,
triggered whenever ssl Article 7 is retrieved.

**Verified:**
- Article 59 now appears in m10's (and m09's) retrieved context.
- **`eval_v3` retrieval re-run again: still bit-identical to baseline**
  (.917/1.000/.953/.967) -- two precondition-mechanism changes this session, zero
  cumulative regression.
- m10 regenerated fresh 3 times: **3/3 pass**, all three explicitly state 17.5%.
- m09 (the English twin, already passing before) re-checked: still passes, now
  also citing Article 59 explicitly for the rate.

**Final behavior-judge tally for this session: 20/20.** Every finding this full
evaluation surfaced -- the mt05/Article-58 conflict, m13/m14's grading target, and
m10's missing rate -- is now fixed and verified, not just documented.